# 05. Lost / Found End-to-End Visual Search — `proyecto_integrador_v2`

Este notebook corresponde al **Paso 05** del pipeline `proyecto_integrador_v2`.

## Objetivo

Probar el pipeline completo con imágenes nuevas de perros perdidos o encontrados.

El flujo end-to-end será:

```text
1. Imagen nueva
2. YOLO detecta perro
3. Se genera crop
4. EfficientNetB0 extrae embedding visual
5. Se normaliza con L2
6. Se compara contra la base de embeddings del Paso 03
7. Se recuperan vecinos Top-K por similitud coseno
8. Se asigna decisión: match fuerte, posible match, débil o sin match claro
```

En este paso, la salida principal no es la raza.  
La salida principal son **posibles coincidencias visuales** contra la base.

## Carpeta de entrada

Coloca imágenes nuevas en:

```text
/content/drive/MyDrive/proyecto_integrador_v2/raw_data/team_test
```

También puedes usar subcarpetas, por ejemplo:

```text
raw_data/team_test/lost/
raw_data/team_test/found/
```

El notebook buscará imágenes en la carpeta `team_test`.

## Archivos necesarios de pasos anteriores

Este notebook requiere:

```text
processed_data/embeddings/step03_dog_embeddings_l2.npy
processed_data/metadata/step03_embeddings_metadata.csv
```

Estos archivos representan la base visual contra la cual se compararán las imágenes que agreguemos

In [ ]:
# 0. Instalación de dependencias

!pip install -q ultralytics tensorflow opencv-python pillow pandas numpy matplotlib tqdm scikit-learn

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import json
import time
import math
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from tqdm import tqdm

from ultralytics import YOLO

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Model

from sklearn.preprocessing import normalize

pd.set_option("display.max_columns", 200)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Montar Google Drive

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto_integrador_v2")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    config = {}

RAW_DATA_PATH = PROJECT_ROOT / "raw_data"
TEAM_TEST_PATH = RAW_DATA_PATH / "team_test"

PROCESSED_DATA_PATH = PROJECT_ROOT / "processed_data"
EMBEDDINGS_PATH = PROCESSED_DATA_PATH / "embeddings"
METADATA_PATH = PROCESSED_DATA_PATH / "metadata"
SEARCH_RESULTS_PATH = PROCESSED_DATA_PATH / "search_results"

STEP05_PATH = PROCESSED_DATA_PATH / "step05_end_to_end_search"
STEP05_CROPS_PATH = STEP05_PATH / "query_crops"
STEP05_RESULTS_PATH = STEP05_PATH / "results"

REPORTS_PATH = PROJECT_ROOT / "reports"
FIGURES_PATH = REPORTS_PATH / "figures"
TABLES_PATH = REPORTS_PATH / "tables"

MODELS_PATH = PROJECT_ROOT / "models"
YOLO_MODELS_PATH = MODELS_PATH / "yolo"
EMBEDDING_MODELS_PATH = MODELS_PATH / "embedding_model"

for p in [
    TEAM_TEST_PATH,
    STEP05_PATH,
    STEP05_CROPS_PATH,
    STEP05_RESULTS_PATH,
    REPORTS_PATH,
    FIGURES_PATH,
    TABLES_PATH,
    YOLO_MODELS_PATH,
    EMBEDDING_MODELS_PATH
]:
    p.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_L2_PATH = EMBEDDINGS_PATH / "step03_dog_embeddings_l2.npy"
EMBEDDINGS_METADATA_PATH = METADATA_PATH / "step03_embeddings_metadata.csv"

TARGET_SIZE = int(config.get("image_size", 224))

detector_config = config.get("detector", {})
YOLO_MODEL_NAME = detector_config.get("default_model", "yolo26s.pt")
FALLBACK_YOLO_MODEL_NAME = detector_config.get("fallback_model", "yolo11s.pt")
CONF_THRESHOLD = float(detector_config.get("confidence_threshold", 0.25))
CROP_MARGIN = float(detector_config.get("crop_margin", 0.15))

TOP_K = 10
VISUAL_TOP_K = 5

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEAM_TEST_PATH:", TEAM_TEST_PATH)
print("EMBEDDINGS_L2_PATH:", EMBEDDINGS_L2_PATH)
print("EMBEDDINGS_METADATA_PATH:", EMBEDDINGS_METADATA_PATH)
print("TARGET_SIZE:", TARGET_SIZE)
print("YOLO_MODEL_NAME:", YOLO_MODEL_NAME)

In [ ]:
# 4. Cargar base de embeddings del Paso 03

if not EMBEDDINGS_L2_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {EMBEDDINGS_L2_PATH}. Ejecuta primero el Paso 03."
    )

if not EMBEDDINGS_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {EMBEDDINGS_METADATA_PATH}. Ejecuta primero el Paso 03."
    )

database_embeddings_l2 = np.load(EMBEDDINGS_L2_PATH)
database_metadata_df = pd.read_csv(EMBEDDINGS_METADATA_PATH).reset_index(drop=True)

if len(database_metadata_df) != database_embeddings_l2.shape[0]:
    print("Advertencia: metadata y embeddings tienen tamaños diferentes.")
    min_len = min(len(database_metadata_df), database_embeddings_l2.shape[0])
    database_metadata_df = database_metadata_df.head(min_len).copy()
    database_embeddings_l2 = database_embeddings_l2[:min_len]

print("Base embeddings:", database_embeddings_l2.shape)
print("Metadata:", database_metadata_df.shape)

display(database_metadata_df.head())

In [ ]:
# 5. Validar normalización de base

norms = np.linalg.norm(database_embeddings_l2, axis=1)

print("Norma mínima:", norms.min())
print("Norma máxima:", norms.max())
print("Norma promedio:", norms.mean())

if np.allclose(norms.mean(), 1.0, atol=1e-3):
    print("OK: base normalizada con L2.")
else:
    print("Advertencia: revisar normalización.")

In [ ]:
# 6. Cargar imágenes nuevas de team_test

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

team_images = [
    p for p in sorted(TEAM_TEST_PATH.rglob("*"))
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

team_test_df = pd.DataFrame({
    "query_image_path": [str(p) for p in team_images],
    "query_filename": [p.name for p in team_images],
    "query_relative_path": [str(p.relative_to(TEAM_TEST_PATH)) for p in team_images]
})

print("Imágenes encontradas en team_test:", len(team_test_df))

if len(team_test_df) == 0:
    print("Coloca imágenes en:", TEAM_TEST_PATH)

display(team_test_df.head())

In [ ]:
# 7. Cargar detector YOLO

def load_yolo_detector(primary_model: str, fallback_model: str):
    try:
        print(f"Intentando cargar detector principal: {primary_model}")
        model = YOLO(primary_model)
        selected_model = primary_model
    except Exception as e:
        print("No se pudo cargar el detector principal.")
        print("Error:", e)
        print(f"Usando detector fallback: {fallback_model}")
        model = YOLO(fallback_model)
        selected_model = fallback_model

    return model, selected_model


yolo_detector, selected_yolo_model = load_yolo_detector(
    YOLO_MODEL_NAME,
    FALLBACK_YOLO_MODEL_NAME
)

dog_class_id = None

for class_id, class_name in yolo_detector.names.items():
    if str(class_name).lower() == "dog":
        dog_class_id = int(class_id)
        break

if dog_class_id is None:
    raise ValueError("No se encontró la clase 'dog' en el modelo YOLO seleccionado.")

print("Modelo YOLO seleccionado:", selected_yolo_model)
print("ID clase dog:", dog_class_id)

In [ ]:
# 8. Crear extractor de embeddings EfficientNetB0

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(TARGET_SIZE, TARGET_SIZE, 3)
)

embedding_model = Model(
    inputs=base_model.input,
    outputs=base_model.output
)

EMBEDDING_MODEL_NAME = "EfficientNetB0_ImageNet_pooling_avg"
EMBEDDING_DIM = int(embedding_model.output_shape[-1])

print("Modelo de embeddings:", EMBEDDING_MODEL_NAME)
print("Dimensión query embedding:", EMBEDDING_DIM)
print("Dimensión base embeddings:", database_embeddings_l2.shape[1])

if EMBEDDING_DIM != database_embeddings_l2.shape[1]:
    raise ValueError(
        f"Dimensión incompatible: query={EMBEDDING_DIM}, base={database_embeddings_l2.shape[1]}"
    )

In [ ]:
# 9. Funciones end-to-end


def read_image_rgb(image_path):
    """Lee una imagen, corrige EXIF y regresa RGB."""
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    return np.array(img)


def detect_dogs_yolo(image_path, model, dog_class_id, conf_threshold=0.25):
    """Detecta perros en una imagen usando YOLO."""
    results = model(str(image_path), conf=conf_threshold, verbose=False)

    detections = []

    for result in results:
        boxes = result.boxes

        if boxes is None:
            continue

        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            if class_id == dog_class_id:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

                w = max(0, x2 - x1)
                h = max(0, y2 - y1)
                area = int(w * h)

                detections.append({
                    "class_id": class_id,
                    "class_name": "dog",
                    "confidence": confidence,
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2),
                    "box_width": int(w),
                    "box_height": int(h),
                    "area": area
                })

    return detections


def select_best_detection(detections):
    """Selecciona la detección principal usando área y confianza."""
    if len(detections) == 0:
        return None

    return sorted(
        detections,
        key=lambda d: (d["area"], d["confidence"]),
        reverse=True
    )[0]


def crop_detection(image_path, detection, output_path, margin=0.15):
    """Genera crop con margen alrededor de la detección."""
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        return None, "image_not_readable"

    img_h, img_w = image_bgr.shape[:2]

    x1, y1, x2, y2 = detection["x1"], detection["y1"], detection["x2"], detection["y2"]

    box_w = x2 - x1
    box_h = y2 - y1

    mx = int(box_w * margin)
    my = int(box_h * margin)

    x1m = max(0, x1 - mx)
    y1m = max(0, y1 - my)
    x2m = min(img_w, x2 + mx)
    y2m = min(img_h, y2 + my)

    if x2m <= x1m or y2m <= y1m:
        return None, "invalid_crop_coordinates"

    crop = image_bgr[y1m:y2m, x1m:x2m]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    ok = cv2.imwrite(str(output_path), crop)

    if not ok:
        return None, "crop_write_failed"

    crop_h, crop_w = crop.shape[:2]

    crop_info = {
        "query_crop_path": str(output_path),
        "query_crop_width": int(crop_w),
        "query_crop_height": int(crop_h),
        "query_crop_x1": int(x1m),
        "query_crop_y1": int(y1m),
        "query_crop_x2": int(x2m),
        "query_crop_y2": int(y2m)
    }

    return crop_info, None


def preprocess_crop_for_embedding(crop_path, target_size=224):
    """Carga un crop y aplica preprocessing de EfficientNet."""
    img = Image.open(crop_path).convert("RGB")
    img = img.resize((target_size, target_size))
    arr = np.array(img).astype(np.float32)
    arr = np.expand_dims(arr, axis=0)
    arr = efficientnet_preprocess(arr)
    return arr


def extract_query_embedding_l2(crop_path, embedding_model):
    """Extrae embedding y normaliza L2."""
    arr = preprocess_crop_for_embedding(crop_path, target_size=TARGET_SIZE)
    emb = embedding_model.predict(arr, verbose=0)
    emb_l2 = normalize(emb, norm="l2")
    return emb[0], emb_l2[0]


def search_topk(query_embedding_l2, database_embeddings_l2, database_metadata_df, top_k=10):
    """Busca Top-K vecinos por similitud coseno."""
    sims = np.dot(query_embedding_l2.reshape(1, -1), database_embeddings_l2.T)[0]

    candidate_indices = np.argpartition(sims, -top_k)[-top_k:]
    top_indices = candidate_indices[np.argsort(sims[candidate_indices])[::-1]]

    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        meta = database_metadata_df.iloc[int(idx)]

        rows.append({
            "neighbor_rank": int(rank),
            "neighbor_index": int(idx),
            "cosine_similarity": float(sims[idx]),
            "neighbor_crop_path": meta.get("crop_path", None),
            "neighbor_original_crop_path": meta.get("original_crop_path", None),
            "neighbor_source_type": meta.get("source_type", None),
            "neighbor_quality_label": meta.get("quality_label", None),
            "neighbor_dog_id": meta.get("dog_id", None),
            "neighbor_report_id": meta.get("report_id", None),
        })

    return pd.DataFrame(rows)


def assign_match_decision(top1_similarity, top5_avg_similarity=None):
    """Asigna decisión inicial basada en similitud visual.

    Estos umbrales son heurísticos y deben calibrarse con identity_test.
    """
    if pd.isna(top1_similarity):
        return "needs_review_no_similarity"

    if top1_similarity >= 0.90:
        return "high_confidence_visual_match"

    if top1_similarity >= 0.80:
        return "possible_visual_match"

    if top1_similarity >= 0.70:
        return "weak_visual_match_needs_review"

    return "no_strong_visual_match"

## Decisión visual

Los umbrales iniciales solo como base:

```text
Top-1 >= 0.90 es high_confidence_visual_match
0.80 - 0.90  es possible_visual_match
0.70 - 0.80  es weak_visual_match_needs_review
< 0.70       es no_strong_visual_match
```

Estos umbrales deberán moverse

In [ ]:
# 10. Ejecutar pipeline end-to-end sobre team_test

query_records = []
neighbor_records = []
embedding_records = []

start = time.time()

for _, row in tqdm(team_test_df.iterrows(), total=len(team_test_df)):
    query_image_path = Path(row["query_image_path"])
    query_filename = row["query_filename"]
    query_relative_path = row["query_relative_path"]

    base_record = {
        "query_image_path": str(query_image_path),
        "query_filename": query_filename,
        "query_relative_path": query_relative_path,
        "selected_yolo_model": selected_yolo_model,
        "conf_threshold": CONF_THRESHOLD,
        "crop_margin": CROP_MARGIN,
        "dog_detected": False,
        "num_dog_detections": 0,
        "best_yolo_confidence": np.nan,
        "query_crop_path": None,
        "query_crop_width": np.nan,
        "query_crop_height": np.nan,
        "embedding_generated": False,
        "top1_similarity": np.nan,
        "top5_avg_similarity": np.nan,
        "final_decision": "not_processed",
        "status": "not_processed",
        "error": None
    }

    try:
        detections = detect_dogs_yolo(
            query_image_path,
            yolo_detector,
            dog_class_id,
            conf_threshold=CONF_THRESHOLD
        )

        base_record["num_dog_detections"] = len(detections)

        best_detection = select_best_detection(detections)

        if best_detection is None:
            base_record["status"] = "needs_review_no_dog_detected"
            base_record["final_decision"] = "needs_review_no_dog_detected"
            query_records.append(base_record)
            continue

        base_record["dog_detected"] = True
        base_record["best_yolo_confidence"] = best_detection["confidence"]

        crop_output_path = STEP05_CROPS_PATH / Path(query_relative_path).with_suffix(".jpg")

        crop_info, crop_error = crop_detection(
            query_image_path,
            best_detection,
            crop_output_path,
            margin=CROP_MARGIN
        )

        if crop_error is not None:
            base_record["status"] = "crop_error"
            base_record["error"] = crop_error
            base_record["final_decision"] = "needs_review_crop_error"
            query_records.append(base_record)
            continue

        base_record.update(crop_info)

        raw_emb, query_emb_l2 = extract_query_embedding_l2(
            crop_info["query_crop_path"],
            embedding_model
        )

        base_record["embedding_generated"] = True

        embedding_records.append({
            "query_image_path": str(query_image_path),
            "query_crop_path": crop_info["query_crop_path"],
            "embedding_dimension": int(len(raw_emb)),
            "embedding_norm_before_l2": float(np.linalg.norm(raw_emb)),
            "embedding_norm_after_l2": float(np.linalg.norm(query_emb_l2))
        })

        neighbors_df = search_topk(
            query_emb_l2,
            database_embeddings_l2,
            database_metadata_df,
            top_k=TOP_K
        )

        neighbors_df["query_image_path"] = str(query_image_path)
        neighbors_df["query_filename"] = query_filename
        neighbors_df["query_crop_path"] = crop_info["query_crop_path"]

        neighbor_records.extend(neighbors_df.to_dict("records"))

        top1_similarity = float(neighbors_df[neighbors_df["neighbor_rank"] == 1]["cosine_similarity"].iloc[0])
        top5_avg_similarity = float(neighbors_df[neighbors_df["neighbor_rank"] <= 5]["cosine_similarity"].mean())

        base_record["top1_similarity"] = top1_similarity
        base_record["top5_avg_similarity"] = top5_avg_similarity
        base_record["final_decision"] = assign_match_decision(
            top1_similarity=top1_similarity,
            top5_avg_similarity=top5_avg_similarity
        )
        base_record["status"] = "searched"

    except Exception as e:
        base_record["status"] = "error"
        base_record["error"] = str(e)
        base_record["final_decision"] = "needs_review_exception"

    query_records.append(base_record)

query_results_df = pd.DataFrame(query_records)
query_neighbors_df = pd.DataFrame(neighbor_records)
query_embeddings_df = pd.DataFrame(embedding_records)

elapsed = (time.time() - start) / 60

print("Pipeline terminado.")
print("Tiempo:", round(elapsed, 2), "minutos")

display(query_results_df)
display(query_neighbors_df.head(20))

In [ ]:
# 11. Guardar resultados del Paso 05

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

QUERY_RESULTS_PATH = STEP05_RESULTS_PATH / "step05_query_results.csv"
QUERY_NEIGHBORS_PATH = STEP05_RESULTS_PATH / "step05_query_neighbors.csv"
QUERY_EMBEDDINGS_PATH = STEP05_RESULTS_PATH / "step05_query_embeddings_summary.csv"

query_results_df.to_csv(QUERY_RESULTS_PATH, index=False)
query_neighbors_df.to_csv(QUERY_NEIGHBORS_PATH, index=False)
query_embeddings_df.to_csv(QUERY_EMBEDDINGS_PATH, index=False)

query_results_df.to_csv(TABLES_PATH / "step05_query_results.csv", index=False)
query_neighbors_df.to_csv(TABLES_PATH / "step05_query_neighbors.csv", index=False)

print("Resultados query guardados en:", QUERY_RESULTS_PATH)
print("Vecinos query guardados en:", QUERY_NEIGHBORS_PATH)
print("Resumen embeddings query guardado en:", QUERY_EMBEDDINGS_PATH)

In [ ]:
# 12. Indicadores finales del Paso 05

total_queries = len(query_results_df)
queries_with_dog = int(query_results_df["dog_detected"].sum()) if total_queries > 0 else 0
queries_searched = int((query_results_df["status"] == "searched").sum()) if total_queries > 0 else 0
queries_errors = int((query_results_df["status"] == "error").sum()) if total_queries > 0 else 0

avg_yolo_conf = float(query_results_df["best_yolo_confidence"].dropna().mean()) if queries_with_dog > 0 else np.nan
avg_top1 = float(query_results_df["top1_similarity"].dropna().mean()) if queries_searched > 0 else np.nan
avg_top5 = float(query_results_df["top5_avg_similarity"].dropna().mean()) if queries_searched > 0 else np.nan

decision_counts = query_results_df["final_decision"].value_counts().to_dict() if total_queries > 0 else {}

step05_indicators_df = pd.DataFrame([
    {"section": "input", "indicator": "total_query_images", "value": total_queries},
    {"section": "detection", "indicator": "queries_with_dog_detected", "value": queries_with_dog},
    {"section": "detection", "indicator": "avg_yolo_confidence", "value": avg_yolo_conf},
    {"section": "search", "indicator": "queries_searched", "value": queries_searched},
    {"section": "search", "indicator": "top_k", "value": TOP_K},
    {"section": "search", "indicator": "database_size", "value": database_embeddings_l2.shape[0]},
    {"section": "similarity", "indicator": "avg_top1_similarity", "value": avg_top1},
    {"section": "similarity", "indicator": "avg_top5_similarity", "value": avg_top5},
    {"section": "status", "indicator": "query_errors", "value": queries_errors},
    {"section": "decision", "indicator": "decision_counts", "value": json.dumps(decision_counts, ensure_ascii=False)},
])

STEP05_INDICATORS_PATH = TABLES_PATH / "step05_end_to_end_indicators.csv"
step05_indicators_df.to_csv(STEP05_INDICATORS_PATH, index=False)

display(step05_indicators_df)
print("Indicadores guardados en:", STEP05_INDICATORS_PATH)

In [ ]:
# 13. Visualización de resultados por query

def resolve_image_path(path_a, path_b=None):
    for p in [path_a, path_b]:
        if p is None or pd.isna(p):
            continue
        p = Path(str(p))
        if p.exists():
            return str(p)
    return None


def show_query_result(query_image_path, top_k=5, save=True):
    result_row = query_results_df[
        query_results_df["query_image_path"] == query_image_path
    ]

    if len(result_row) == 0:
        print("Query no encontrado:", query_image_path)
        return

    result_row = result_row.iloc[0]

    if result_row["status"] != "searched":
        print("Query no procesado para búsqueda:")
        display(result_row)
        return

    neighbors = (
        query_neighbors_df[
            (query_neighbors_df["query_image_path"] == query_image_path) &
            (query_neighbors_df["neighbor_rank"] <= top_k)
        ]
        .sort_values("neighbor_rank")
    )

    query_crop_path = result_row["query_crop_path"]

    image_paths = [query_crop_path]
    titles = [
        f"Query\\nTop1={result_row['top1_similarity']:.3f}\\n{result_row['final_decision']}"
    ]

    for _, row in neighbors.iterrows():
        neighbor_path = resolve_image_path(
            row.get("neighbor_original_crop_path", None),
            row.get("neighbor_crop_path", None)
        )
        image_paths.append(neighbor_path)
        titles.append(
            f"Rank {int(row['neighbor_rank'])}\\n"
            f"cos={row['cosine_similarity']:.3f}"
        )

    cols = len(image_paths)
    plt.figure(figsize=(3.2 * cols, 3.8))

    for i, img_path in enumerate(image_paths):
        plt.subplot(1, cols, i + 1)

        if img_path is None:
            plt.text(0.5, 0.5, "Imagen no encontrada", ha="center", va="center")
            plt.axis("off")
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            plt.imshow(img)
            plt.title(titles[i], fontsize=9)
            plt.axis("off")
        except Exception as e:
            plt.text(0.5, 0.5, f"Error\\n{e}", ha="center", va="center")
            plt.axis("off")

    plt.tight_layout()

    if save:
        safe_name = Path(query_image_path).stem
        fig_path = FIGURES_PATH / f"step05_query_{safe_name}_neighbors.png"
        plt.savefig(fig_path, dpi=150)
        print("Figura guardada en:", fig_path)

    plt.show()


if len(query_results_df) > 0:
    searched_queries = query_results_df[query_results_df["status"] == "searched"]["query_image_path"].tolist()

    for q in searched_queries:
        show_query_result(q, top_k=VISUAL_TOP_K, save=True)
else:
    print("No hay queries para visualizar.")